# Attention Rollout para Transformers

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Mapas de atenção de uma única camada são ruidosos. *Attention rollout* multiplica as matrizes de atenção camada a camada (considerando a conexão residual) para estimar quanto cada token de entrada contribuiu para um token de saída depois de todas as camadas.


## Formulação Matemática

$$\tilde A^{(l)} = 0.5 \big(\bar A^{(l)} + I\big)$$
$$R = \tilde A^{(L)} \cdots \tilde A^{(2)}\,\tilde A^{(1)}$$

$\bar A^{(l)}$ é a média das cabeças de atenção na camada $l$; o termo $I$ modela o stream residual.


## Implementação


In [ ]:
import torch


In [ ]:
def attention_rollout(attn_layers):
    """attn_layers: list of (N, N) tensors (head-averaged attention per layer)."""
    N = attn_layers[0].size(-1)
    R = torch.eye(N)
    for A in attn_layers:
        A = 0.5 * (A + torch.eye(N))
        A = A / A.sum(-1, keepdim=True)
        R = A @ R
    return R


## Experimento


In [ ]:
torch.manual_seed(0)
# Fake 3 layers of attention for 6 tokens
attn_layers = [torch.softmax(torch.randn(6, 6), dim=-1) for _ in range(3)]
R = attention_rollout(attn_layers)
print('rollout matrix:', R.shape)
print('token 0 attends most to:', R[0].argmax().item())


## Discussão

- Rollout é *heurística*; métodos baseados em gradiente (atenção × gradiente) costumam dar atribuições mais limpas em classificação.
- Para modelos BERT-like uma variante comum adiciona `1` na diagonal antes da mistura (já feito acima).
- Visualize sobrepondo valores de rollout sobre o texto — útil para mostrar a evidência do modelo.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
